In [26]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
import os
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm import tqdm
import snntorch as snn
import time
import subprocess
import re
import thop
from snntorch import surrogate
from snntorch import spikegen
import torch.nn.functional as F

In [2]:
DATA_DIR = "../../data"
DATASET_DIR = f"{DATA_DIR}/processed"

In [3]:
class CustomDatasetSpike(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, num_steps=4):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.num_steps = num_steps

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)
            label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
            img = spikegen.rate(img, num_steps=self.num_steps, gain=1)
            return img, label


class CustomDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)
            label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
            return img, label

In [4]:
test_df = pd.read_csv(f"{DATASET_DIR}/test1.csv")     
test_df_multi = pd.read_csv(f"{DATASET_DIR}/test1_multi.csv")   

In [5]:
LE = LabelEncoder()
LE_multi = LabelEncoder()
BATCH_SIZE = 40

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

LE = LabelEncoder()
LE.fit(test_df["Label"])

# LE.classes_

# swap classes in the label encoder
swapped_classes = LE.classes_.copy()
swapped_classes[0], swapped_classes[1] = swapped_classes[1], swapped_classes[0]

LE.classes_ = swapped_classes

test_df_encoded = test_df.copy()
test_df_encoded["Label"] = LE.transform(test_df_encoded["Label"])

# Multi
LE_multi.fit(test_df_multi["Label"])
test_df_multi_encoded = test_df_multi.copy()
test_df_multi_encoded["Label"] = LE_multi.transform(test_df_multi_encoded["Label"])


In [6]:
class BinaryCNN3(nn.Module):
    def __init__(self, num_classes=2):
        super(BinaryCNN3, self).__init__()
        
        # Feature extraction layers
        self.conv1 = nn.Sequential(
            # First block: 32x32 -> 16x16
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3),
            
        )

        self.conv2 = nn.Sequential(
            # Second block: 16x16 -> 8x8
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3)
        )

        
        # Binary classifier
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8 , 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        x = self.conv1(x)
        # print(x.shape)
        x = self.conv2(x)
        # print(x.shape)
        x = torch.flatten(x, 1)
        # print(x.shape)
        return self.classifier(x)

In [31]:
class BasicSNN(nn.Module):
    def __init__(self, beta=1, num_steps=4, num_classes=2):
        super(BasicSNN, self).__init__()

        self.num_steps = num_steps
        self.num_classes = num_classes
        self.spike_grad = surrogate.fast_sigmoid(slope=25)

        self.conv1 = nn.Conv2d(1, 16, kernel_size=4, stride=4)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=self.spike_grad, threshold=0.3)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=2, stride=2)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=self.spike_grad, threshold=0.3)


        # flattened_size = 32 * (32 // 4) * (32 // 4)

        self.fc1 = nn.Linear(32, self.num_classes)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=self.spike_grad, threshold=0.3)

        self.spk1 = None
        self.spk2 = None


    def forward(self, x):
        # x shape: [batch_size, num_steps, C, H, W]
        x = x.permute(1, 0, 2, 3, 4)  # New shape: [num_steps, batch_size, C, H, W]
        
        mem1 = self.lif1.reset_mem()
        mem2 = self.lif2.reset_mem()
        mem3 = self.lif3.reset_mem()

        spk_rec = []
        mem_rec = []

        spk1_rec = []
        spk2_rec = []

        for step in range(self.num_steps):
            x_step = x[step]  # [batch_size, C, H, W] for current timestep

            # Layer 1
            cur1 = self.conv1(x_step)
            cur1 = F.max_pool2d(cur1, 2)  # Optional pooling
            spike1, mem1 = self.lif1(cur1, mem1)

            # Layer 2
            cur2 = self.conv2(spike1)
            cur2 = F.max_pool2d(cur2, 2)  # Optional pooling
            spike2, mem2 = self.lif2(cur2, mem2)

            # print(spike2.shape)
            # print(spike2.flatten(1).shape)

            # Classifier
            cur3 = self.fc1(spike2.flatten(1))
            spike3, mem3 = self.lif3(cur3, mem3)


            spk1_rec.append(spike1)
            spk2_rec.append(spike2)

            spk_rec.append(spike3)
            mem_rec.append(mem3)

        self.spk1 = torch.stack(spk1_rec, dim=0)
        self.spk2 = torch.stack(spk2_rec, dim=0)

        return torch.stack(spk_rec, dim=0), torch.stack(mem_rec, dim=0)

In [21]:
# Binary Classification
test_data_sample = test_df_encoded.groupby("Label").sample(6000, random_state=42)

# shuffle the data
test_data_sample = test_data_sample.sample(frac=1, random_state=42)
test_dataset = CustomDataset(test_data_sample, f"{DATASET_DIR}/images", transform=transform)
test_dataset_spike = CustomDatasetSpike(test_data_sample, f"{DATASET_DIR}/images", transform=transform)


# Multi Classification
test_data_sample_multi = test_df_multi_encoded.groupby("Label").sample(2000, random_state=42)

# shuffle the data
test_data_sample_multi = test_data_sample_multi.sample(frac=1, random_state=42)
test_dataset_multi = CustomDataset(test_data_sample_multi, f"{DATASET_DIR}/images", transform=transform)

In [22]:
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader_spike = DataLoader(test_dataset_spike, batch_size=BATCH_SIZE, shuffle=False)
test_loader_multi = DataLoader(test_dataset_multi, batch_size=BATCH_SIZE, shuffle=False)

In [39]:
def calculate_metrics(model_class, weight_path, test_dataloader, device='cuda', 
                     is_snn=False, model_params={}):
    """
    Calculate metrics for both CNN and SNN models with NVIDIA SMI energy measurement.
    """
    # Load model and weights
    saved_info = torch.load(weight_path, weights_only=False)
    model = model_class(**model_params)
    model.load_state_dict(saved_info['model_state_dict'])
    model.to(device)
    model.eval()

    # 1. Parameter Count
    num_params = sum(p.numel() for p in model.parameters())

    # 2. FLOPs/SynOps Calculation
    flops = 0
    synops = 0
    
    if is_snn:
        # SNN-specific calculation
        with torch.no_grad():
            for inputs, _ in tqdm(test_dataloader):
                inputs = inputs.to(device)

                spk_rec, _ = model(inputs)
                
                # Calculate SynOps (sum of all spikes)
                synops += spk_rec.sum().item()
        
        # Estimate base FLOPs (non-spiking operations)
        inputs, _ = next(iter(test_dataloader))
        flops, _ = thop.profile(model, inputs=(inputs.to(device),), verbose=False)
    else:
        # CNN FLOPs calculation
        inputs, _ = next(iter(test_dataloader))
        flops, _ = thop.profile(model, inputs=(inputs.to(device),), verbose=False)

    # 3. Execution Speed
    start_time = time.time()
    with torch.no_grad():
        for inputs, _ in test_dataloader:
            inputs = inputs.to(device)
            _ = model(inputs)
    exec_time = time.time() - start_time
    exec_speed = len(test_dataloader.dataset) / exec_time

    # 4. Energy Consumption using NVIDIA SMI
    def get_power():
        result = subprocess.run(['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
                               stdout=subprocess.PIPE)
        return float(re.search(r'\d+\.\d+', result.stdout.decode()).group())
    
    # Warm-up
    _ = model(inputs.to(device))
    
    # Measure power
    start_power = get_power()
    start_energy = time.time()
    with torch.no_grad():
        for inputs, _ in test_dataloader:
            inputs = inputs.to(device)
            _ = model(inputs)
    
    duration = time.time() - start_energy
    avg_power = (get_power() + start_power) / 2
    energy = avg_power * duration  # Convert watts to joules over time
    energy_per_12k = (energy / len(test_dataloader.dataset)) * 12000

    return {
        'parameters': num_params,
        'flops': flops if not is_snn else flops * 4,
        'synops': synops if is_snn else 0,
        'execution_speed': exec_speed,
        'energy_per_12k': energy_per_12k
    }

def valid_comparison(cnn_metrics, snn_metrics):
    """Generate meaningful comparison between SNN and CNN metrics"""
    return {
        'parameter_ratio': snn_metrics['parameters'] / cnn_metrics['parameters'],
        'flops_vs_synops': {
            'cnn_flops': cnn_metrics['flops'],
            'snn_flops': snn_metrics['flops'],
            'snn_synops': snn_metrics['synops']
        },
        'speed_ratio': snn_metrics['execution_speed'] / cnn_metrics['execution_speed'],
        'energy_ratio': snn_metrics['energy_per_12k'] / cnn_metrics['energy_per_12k']
    }

In [9]:
base_dir = "../../models/checkpoints"

available_dirs = {
    0 : "cnn",
    1 : "multi_cnn",
    2 : "basic_snn",
    3 : "basic_multisnn",
    4 : "paper_snn",
    5 : "paper_multisnn"
}

In [40]:
calculate_metrics(BasicSNN, f"{base_dir}/basic_snn/model_ratev1bg1.pt", test_loader_spike, device='cuda', is_snn=True, model_params={'num_steps': 4, 'beta': 1})

100%|██████████| 300/300 [00:27<00:00, 10.83it/s]


{'parameters': 2418,
 'flops': 15769600.0,
 'synops': 47998.0,
 'execution_speed': 498.6158637095748,
 'energy_per_12k': 234.7682604122162}

In [16]:
calculate_metrics(BinaryCNN3, f"{base_dir}/cnn/bcnn3_crossentropy_plateau.pt", test_loader, device='cuda', is_snn=False, model_params={})

100%|██████████| 300/300 [00:18<00:00, 15.84it/s]


14.9


100%|██████████| 300/300 [00:16<00:00, 18.02it/s]


{'parameters': 138178,
 'flops': 62343680.0,
 'synops': 0,
 'execution_speed': 633.5905545353427,
 'energy_per_12k': 239.67271529912952}